# Практика · ML Handbook гл. 44 (compression)

Источник: [nn-optimization.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/nn-optimization.html)

Теория: [../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md](../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md) · [README.md](./README.md)

**Handbook-код — в этом файле.** Эталон: [handbook_gl44_практика_решения.ipynb](./handbook_gl44_практика_решения.ipynb) **после** попытки.

## Зачем практику handbook

Закрепить главу **руками**: перед каждым блоком — **Зачем этот фрагмент** и ссылка на конспект; после запуска — **Что проверить** по `print`.

| ☐ | Блок |
|---|------|
| ☐ | §44.3 distill |
| ☐ | §44.2 prune |
| ☐ | §44.1 quant |


## Импорты

Один раз в начале.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# §44.3 · Distillation

## Зачем этот фрагмент

Ученик учится и по жёсткой разметке, и по «мягким» вероятностям учителя с температурой T.

См. [d.5.44 §4](../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md#4-дистилляция-handbook-443).

## Что проверить

Скалярный loss без NaN; обычно > 0.


In [ ]:
def distill(s, t, y, T=4.0, alpha=0.3):
    hard = F.cross_entropy(s, y)
    soft = F.kl_div(F.log_softmax(s/T,1), F.softmax(t/T,1), reduction='batchmean') * T*T
    return alpha*hard + (1-alpha)*soft
s = torch.randn(8, 10, requires_grad=True); t = torch.randn(8, 10); y = torch.randint(0,10,(8,))
print('distill:', round(distill(s,t,y).item(), 3))


---


# §44.2 · Prune

## Зачем этот фрагмент

Обнуляешь долю самых мелких |w| — первый шаг к разреженной сети без смены архитектуры.

См. [d.5.44 §3](../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md#3-прунинг-handbook-442).

## Что проверить

Sparsity около **60%** нулей в `weight`.


In [ ]:
import torch.nn.utils.prune as prune
layer = nn.Linear(100, 100)
prune.l1_unstructured(layer, name='weight', amount=0.6)
print('sparsity:', f'{(layer.weight==0).float().mean():.0%}')


---


# §44.1 · quantize_dynamic

## Зачем этот фрагмент

Динамическое квантование Linear на CPU — быстрый выигрыш на инференсе без полного PTQ-пайплайна.

См. [d.5.44 §2](../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md#2-квантование-handbook-441).

## Что проверить

Тип модуля сменился на квантованный wrapper.


In [ ]:
import torch.ao.quantization as tq
m = nn.Sequential(nn.Linear(32,16), nn.ReLU(), nn.Linear(16,10))
print('quant:', type(tq.quantize_dynamic(m, {nn.Linear}, dtype=torch.qint8)))
